In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/cafa-6-protein-function-prediction/sample_submission.tsv
/kaggle/input/cafa-6-protein-function-prediction/IA.tsv
/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta
/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset-taxon-list.tsv
/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv
/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta
/kaggle/input/cafa-6-protein-function-prediction/Train/train_taxonomy.tsv
/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo


In [2]:
# ======================================================
# CAFA-6 LIGHTGBM PIPELINE (FINAL • MEMORY SAFE • STABLE)
# ======================================================

!pip install -q lightgbm biopython networkx

# ======================================================
# IMPORTS
# ======================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
import networkx as nx
from Bio import SeqIO
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer


# ======================================================
# PATHS
# ======================================================
BASE = "/kaggle/input/cafa-6-protein-function-prediction"

TRAIN_TERMS = f"{BASE}/Train/train_terms.tsv"
TRAIN_FASTA = f"{BASE}/Train/train_sequences.fasta"
TEST_FASTA  = f"{BASE}/Test/testsuperset.fasta"
SAMPLE_SUB  = f"{BASE}/sample_submission.tsv"
GO_OBO_FILE = f"{BASE}/Train/go-basic.obo"


# ======================================================
# LOAD LABELS
# ======================================================
print("Loading labels...")

train_terms = pd.read_csv(TRAIN_TERMS, sep="\t")

# optional: remove CC
train_terms = train_terms[train_terms["aspect"] != "CC"]

grouped = train_terms.groupby("EntryID")["term"].apply(list)

train_ids = grouped.index.values
labels = grouped.values

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(labels).astype(np.uint8)   # save RAM
go_terms = mlb.classes_

print("Y shape:", Y.shape)


# ======================================================
# LOAD TEST IDS
# ======================================================
sample_sub = pd.read_csv(
    SAMPLE_SUB,
    sep="\t",
    header=None,
    names=["EntryID","term","score"],
    on_bad_lines="skip"
)

test_ids = sample_sub["EntryID"].unique()


# ======================================================
# FEATURE ENGINEERING
# 20 amino-acid frequency features
# ======================================================
AA = "ACDEFGHIKLMNPQRSTVWY"
AA_INDEX = {a:i for i,a in enumerate(AA)}

def seq_to_feat(seq):
    v = np.zeros(20, dtype=np.float32)
    for c in seq:
        if c in AA_INDEX:
            v[AA_INDEX[c]] += 1
    return v / max(len(seq),1)


# ======================================================
# LOAD SEQUENCES
# ======================================================
print("Loading sequences...")

train_seq = {
    (r.id.split("|")[1] if "|" in r.id else r.id): str(r.seq)
    for r in SeqIO.parse(TRAIN_FASTA, "fasta")
}

test_seq = {
    (r.id.split("|")[1] if "|" in r.id else r.id): str(r.seq)
    for r in SeqIO.parse(TEST_FASTA, "fasta")
}


# ======================================================
# ALIGN TRAIN
# ======================================================
print("Aligning train...")

keep = [i for i,p in enumerate(train_ids) if p in train_seq]

train_ids = train_ids[keep]
Y = Y[keep]


# ======================================================
# BUILD TRAIN FEATURES
# ======================================================
print("Extracting train features...")
X = np.vstack([seq_to_feat(train_seq[p]) for p in train_ids])


# ======================================================
# ALIGN TEST + BUILD TEST FEATURES
# ======================================================
test_ids = np.array([p for p in test_ids if p in test_seq])

print("Extracting test features...")
X_test = np.vstack([seq_to_feat(test_seq[p]) for p in test_ids])


print("Shapes:")
print("X:", X.shape)
print("Y:", Y.shape)
print("X_test:", X_test.shape)


# ======================================================
# LIGHTGBM (STREAMING • MEMORY SAFE)
# ======================================================
# DO NOT store models → train → predict → delete
# ======================================================

MAX_GO = 1000   # ⭐ smaller = faster + safer (recommended)

print("Selecting top frequent GO terms...")

freq = Y.sum(axis=0)
top_idx = np.argsort(freq)[-MAX_GO:]

Y_small = Y[:, top_idx]
go_terms_small = go_terms[top_idx]

NUM_GO = Y_small.shape[1]

params = {
    "objective":"binary",
    "learning_rate":0.05,
    "num_leaves":31,
    "verbosity":-1
}

print("Training + predicting (streaming)...")

probs = np.zeros((len(test_ids), NUM_GO), dtype=np.float32)

for j in tqdm(range(NUM_GO)):

    y_col = Y_small[:,j]

    if y_col.sum() < 5:
        continue

    dtrain = lgb.Dataset(X, label=y_col)

    model = lgb.train(params, dtrain, num_boost_round=50)

    probs[:,j] = model.predict(X_test)

    del model   # ⭐ frees RAM immediately


# ======================================================
# WRITE submission.tsv
# ======================================================
print("Writing submission.tsv...")

TOP_K = 50

with open("submission.tsv","w") as f:
    for i,pid in enumerate(test_ids):
        row = probs[i]
        idx = np.argsort(row)[-TOP_K:][::-1]
        for j in idx:
            if row[j] > 0:
                f.write(f"{pid}\t{go_terms_small[j]}\t{row[j]:.4f}\n")


# ======================================================
# YOUR FINAL STEP (UNCHANGED)
# ======================================================
def load_go_dag(obo_file):
    dag = nx.DiGraph()
    current=None
    with open(obo_file) as f:
        for line in f:
            line=line.strip()
            if line=="[Term]":
                current=None
            elif line.startswith("id: "):
                current=line.split()[1]
                dag.add_node(current)
            elif current and line.startswith("is_a: "):
                dag.add_edge(current,line.split()[1])
            elif current and line.startswith("relationship: part_of "):
                dag.add_edge(current,line.split()[2])
    topo=list(nx.topological_sort(dag))
    parents={n:list(dag.successors(n)) for n in dag.nodes}
    return topo,parents


topo_terms, parent_map = load_go_dag(GO_OBO_FILE)


submission_df = pd.read_csv(
    "submission.tsv",
    sep="\t",
    header=None,
    names=["protein_id","go_term","score"],
    dtype={"score":np.float32},
    on_bad_lines="skip"
)

print(submission_df.head())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.2 MB/s eta 0:00:00
Loading labels...
Y shape: (82404, 26125)
Loading sequences...
Aligning train...
Extracting train features...
Extracting test features...
Shapes:
X: (82404, 20)
Y: (82404, 26125)
X_test: (10000, 20)
Selecting top frequent GO terms...
Training + predicting (streaming)...


100%|██████████| 1000/1000 [06:27<00:00,  2.58it/s]


Writing submission.tsv...
   protein_id     go_term   score
0  A0A0C5B5G6  GO:0001649  0.5908
1  A0A0C5B5G6  GO:0140297  0.3575
2  A0A0C5B5G6  GO:0005515  0.3450
3  A0A0C5B5G6  GO:0005739  0.3033
4  A0A0C5B5G6  GO:0005743  0.2641
